<a href="https://colab.research.google.com/github/souvikkai/souvik-ai-pm-portfolio/blob/main/day16-long-context/Day16_long_context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install anthropic -q
import anthropic, time, json
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
MODEL = "claude-haiku-4-5-20251001"
INPUT_COST  = 0.0008   # per 1K tokens
OUTPUT_COST = 0.004    # per 1K tokens

def cost(inp, out):
    return round((inp/1000)*INPUT_COST + (out/1000)*OUTPUT_COST, 6)

print("✅ Ready. Model:", MODEL)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 33.2 MB/s eta 0:00:00
✅ Ready. Model: claude-haiku-4-5-20251001


In [2]:
# Base document — AI infra content, realistic text
BASE = """The H100 SXM5 GPU from NVIDIA delivers 3.35 TB/s HBM3 bandwidth
and 80GB memory capacity. It is designed for large language model training
and inference workloads. The GPU features 132 streaming multiprocessors and
supports NVLink for multi-GPU configurations. For serving Llama 70B at FP16
precision, two H100s are required due to the 140GB memory footprint.
At Lambda Labs, the H100 is priced at $2.49 per hour per GPU. """

# Build 3 context sizes by repeating base document
def build_context(target_tokens):
    # ~4 chars per token estimate
    target_chars = target_tokens * 4
    text = (BASE * (target_chars // len(BASE) + 1))[:target_chars]
    return text

SIZES = {"1K": 1000, "10K": 10000, "100K": 100000}
contexts = {k: build_context(v) for k, v in SIZES.items()}

QUESTION = "Based on the above, what is the hourly cost of H100 on Lambda Labs?"

for k, v in contexts.items():
    print(f"{k} context: ~{len(v)//4} tokens, {len(v):,} chars")

1K context: ~1000 tokens, 4,000 chars
10K context: ~10000 tokens, 40,000 chars
100K context: ~100000 tokens, 400,000 chars


In [3]:
results = []

for label, context in contexts.items():
    print(f"\nRunning {label} context...", end=" ")

    prompt = context + "\n\n" + QUESTION

    t_start = time.time()
    response = client.messages.create(
        model=MODEL,
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    )
    latency = round(time.time() - t_start, 2)

    inp   = response.usage.input_tokens
    out   = response.usage.output_tokens
    c     = cost(inp, out)

    results.append({
        "context_size": label,
        "input_tokens": inp,
        "output_tokens": out,
        "latency_sec": latency,
        "cost_usd": c,
        "answer": response.content[0].text.strip()
    })

    print(f"✅ {inp} tokens | {latency}s | ${c:.5f}")
    time.sleep(1)

print("\nDone.")


Running 1K context... ✅ 1265 tokens | 1.35s | $0.00140

Running 10K context... ✅ 12389 tokens | 1.15s | $0.01031

Running 100K context... ✅ 123677 tokens | 2.59s | $0.09933

Done.


In [4]:
print("="*65)
print(f"{'Context':>10} {'Input Tok':>12} {'Latency(s)':>12} {'Cost':>10} {'Cost×100K/day':>15}")
print("-"*65)

base_latency = results[0]["latency_sec"]
base_cost    = results[0]["cost_usd"]

for r in results:
    daily = r["cost_usd"] * 100_000
    print(f"{r['context_size']:>10} {r['input_tokens']:>12,} {r['latency_sec']:>12.2f} "
          f"${r['cost_usd']:>9.5f} ${daily:>13,.2f}")

print("="*65)
print()
print("SCALING ANALYSIS:")
for r in results[1:]:
    tok_mult = round(r['input_tokens'] / results[0]['input_tokens'], 1)
    lat_mult = round(r['latency_sec'] / results[0]['latency_sec'], 1)
    cost_mult = round(r['cost_usd'] / results[0]['cost_usd'], 1)
    print(f"  {r['context_size']} vs 1K: tokens {tok_mult}x | latency {lat_mult}x | cost {cost_mult}x")

print()
print("ANSWERS (quality check — should all say $2.49/hr):")
for r in results:
    print(f"  [{r['context_size']}] {r['answer'][:80]}")

print()
print("KEY INSIGHT:")
print("  Does latency scale linearly with context? Compare token multiplier vs latency multiplier.")
print("  If latency grows faster than tokens → superlinear (attention is O(n²))")
print("  If latency grows same as tokens → linear (mostly IO bound, HBM bandwidth)")

   Context    Input Tok   Latency(s)       Cost   Cost×100K/day
-----------------------------------------------------------------
        1K        1,265         1.35 $  0.00140 $       140.00
       10K       12,389         1.15 $  0.01031 $     1,031.10
      100K      123,677         2.59 $  0.09933 $     9,932.60

SCALING ANALYSIS:
  10K vs 1K: tokens 9.8x | latency 0.9x | cost 7.4x
  100K vs 1K: tokens 97.8x | latency 1.9x | cost 70.9x

ANSWERS (quality check — should all say $2.49/hr):
  [1K] # H100 Hourly Cost on Lambda Labs

Based on the information provided, the **hour
  [10K] # Hourly Cost of H100 on Lambda Labs

Based on the information provided, the hou
  [100K] # Hourly Cost of H100 on Lambda Labs

According to the information provided, the

KEY INSIGHT:
  Does latency scale linearly with context? Compare token multiplier vs latency multiplier.
  If latency grows faster than tokens → superlinear (attention is O(n²))
  If latency grows same as tokens → linear (mostly IO bou